# Exploratory Data Analysis: feat_churn

This notebook performs hypothesis/business-question-driven EDA on the final feature-engineered table (`feat_churn`), produced by `retention_platform.features.build`.

**Scope:** this is deliberately focused, not exhaustive analysis of every column in `feat_churn`. It is organized around specific business questions and the three engineered-feature hypotheses from Commit 3, not a mechanical pass over all 47 columns. Anything interesting found outside
this planned scope is noted as a candidate for a later commit rather than acted on here.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

from retention_platform.config import load_config

db_path = load_config()["paths"]["interim_db"]
conn = duckdb.connect(str(db_path))
df = conn.execute("SELECT * FROM feat_churn").fetchdf()
conn.close()

df.shape

## 1. Target Distribution

In [ ]:
counts = df["is_voluntary_churn"].value_counts()
percentages = df["is_voluntary_churn"].value_counts(normalize=True) * 100

summary = pd.DataFrame({"count": counts, "percent": percentages.round(2)})
summary

In [ ]:
counts_sorted = counts.sort_index()

fig, ax = plt.subplots(figsize=(6, 6))

counts_sorted.plot(kind="bar", ax=ax, color=["#4C72B0", "#DD8452"])

ax.set_xlabel("is_voluntary_churn")
ax.set_ylabel("count")
ax.set_title("Target Distribution: Voluntary Churn")
ax.set_xticklabels(["False", "True"], rotation=0)

for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{value / counts_sorted.sum() * 100:.1f}%" for value in counts_sorted],
        padding=3,
    )

plt.tight_layout()
plt.show()

In [ ]:
churn_rate = df["is_voluntary_churn"].mean() * 100
print(f"Voluntary churn rate: {churn_rate:.2f}%")

**Observations:**

- Voluntary churners make up 26.47% of the dataset (1,863 of 7,037 customers), a clear minority class.
- Retained customers account for the remaining 73.53%.

**Conclusions:**

- A classifier that predicted "did not churn" for every customer would already score about 73.5% on `Accuracy` without ever flagging a single at-risk customer, so `Accuracy` is a poor primary metric here.
- This is why the locked Business Design's primary metrics are `Precision@K`, `Lift@K`, and `PR-AUC` rather than `Accuracy` - they evaluate how well the model ranks and identifies the minority class, which is what actually matters for a targeted outreach use case.

## 2. Univariate Distributions of Key Numeric Features

In [ ]:
numeric_features = [
    "tenure_in_months",
    "monthly_charge",
    "total_charges",
    "num_addon_services",
    "satisfaction_score",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, numeric_features):
    ax.hist(df[col].dropna(), bins=30, color="#4C72B0", edgecolor="white")
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("count")

for ax in axes[len(numeric_features):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

**Observations:**

- **tenure_in_months:** a large spike at the low end (many customers in their first few months), then a roughly declining spread across the rest of the range, with a secondary bump near the maximum tenure. This bump is not explained further here — noted as a candidate for Commit 5 (e.g., checking whether it reflects a fixed dataset observation window rather than a genuine customer behavior pattern).
- **monthly_charge:** a broad, somewhat multi-modal spread rather than a clean bell shape, consistent with distinct pricing tiers for different service bundles.
- **total_charges:** strongly right-skewed, which follows directly from its relationship to tenure (customers with short tenure cannot have accumulated large total charges yet).
- **num_addon_services:** spread across the 0-8 range with a visible peak at 0 (no add-ons), rather than concentrated near either extreme.
- **satisfaction_score:** takes a small number of discrete values (1-5) and shows a roughly central concentration rather than being uniform across the range.

## 3. Feature vs. Churn Relationships

In [ ]:
categorical_features = ["contract", "internet_type", "payment_method"]

for col in categorical_features:
    churn_by_category = (
        df.groupby(col)["is_voluntary_churn"]
        .mean()
        .sort_values(ascending=False)
        * 100
    )

    print(f"Churn rate (%) by {col}:")
    print(churn_by_category.round(2))
    print()

    fig, ax = plt.subplots(figsize=(6, 6))

    churn_by_category.plot(kind="bar", ax=ax, color="#DD8452")

    ax.set_ylabel("voluntary churn rate (%)")
    ax.set_title(f"Voluntary Churn Rate by {col}")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=3)

    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

**Observations:**

- **contract:** churn rate is highest for Month-to-Month customers (45.75%), then One Year (10.71%), then Two Year (2.55%) - churn drops sharply as contract length increases.
- **internet_type:** Fiber Optic customers churn the most (40.65%), followed by Cable (25.66%) and DSL (18.48%).
- **payment_method:** Mailed Check (36.88%) and Bank Withdrawal (33.90%) customers churn at a notably higher rate than Credit Card customers (14.48%).

**Conclusions:**

- Contract length shows the strongest and most monotonic relationship with churn of the three features.
- Fiber Optic customers churning more than DSL/Cable customers is not explained further here — noted as a candidate for Commit 5 (e.g., checking price or reliability differences by internet type) rather than assumed in this notebook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, col in zip(axes, ["tenure_in_months", "monthly_charge"]):
    churned = df.loc[df["is_voluntary_churn"], col].dropna()
    stayed = df.loc[~df["is_voluntary_churn"], col].dropna()
    ax.hist(stayed, bins=30, alpha=0.5, label="not voluntary churn", color="#4C72B0", density=True)
    ax.hist(churned, bins=30, alpha=0.5, label="voluntary churn", color="#DD8452", density=True)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("density")
    ax.legend()

plt.tight_layout()
plt.show()

**Observations:**

- **tenure_in_months:** voluntary churners are visibly concentrated at low tenure, while non-churners are spread more evenly across the range, including many long-tenured customers.
- **monthly_charge:** churners skew toward higher monthly charges than non-churners.

**Conclusions:**

- The tenure pattern is directionally plausible - newer customers have had less time to become invested in the service and are more likely to leave early.
- The monthly_charge pattern is also plausible - customers paying more may be more price-sensitive to competitor offers, consistent with "Competitor" being the leading churn reason found in the Stage 2 dataset-understanding notebook.

## 4. Checking Engineered-Feature Hypotheses

**Hypothesis (num_addon_services):** more add-ons correlates with lower churn, on the reasoning that more add-ons represent a higher switching cost.

In [ ]:
churn_by_addons = df.groupby("num_addon_services")["is_voluntary_churn"].mean() * 100
print("Churn rate (%) by num_addon_services:")
print(churn_by_addons.round(2))

fig, ax = plt.subplots(figsize=(6, 6))
churn_by_addons.plot(kind="bar", ax=ax, color="#55A868")
ax.set_xlabel("num_addon_services")
ax.set_ylabel("voluntary churn rate (%)")
ax.set_title("Voluntary Churn Rate by Number of Add-on Services")

# Add percentage labels on top of each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)
    
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Observations:**

- From 1 add-on through 8 add-ons, churn rate declines steadily and substantially, from 50.22% down to 4.89%.
- The 0-add-ons group does not fit that pattern - it has the lowest churn rate of all (9.40%), even lower than customers with several add-ons.

**Conclusions:**

- Within customers who have internet service, the switching-cost hypothesis holds: more add-ons correlates with lower churn.
- The 0-add-ons group likely overlaps heavily with customers who have no internet service at all (phone-only customers), a different population than "internet customers who declined add-ons," which breaks the simple monotonic story across the full 0-8 range.
- **Verdict: partially supported.** Noted here as a candidate for Commit 5 (e.g. conditioning this feature on internet_service) rather than acted on in this notebook.

**Hypothesis (is_month_to_month):** month-to-month customers churn at a much higher rate than customers on longer contracts, since they have no
switching friction (no contract term to complete before leaving).

In [ ]:
churn_by_mtm = df.groupby("is_month_to_month")["is_voluntary_churn"].mean() * 100

print("Churn rate (%) by is_month_to_month:")
print(churn_by_mtm.round(2))

fig, ax = plt.subplots(figsize=(6, 6))

churn_by_mtm.plot(kind="bar", ax=ax, color="#C44E52")

ax.set_xlabel("is_month_to_month")
ax.set_ylabel("voluntary churn rate (%)")
ax.set_title("Voluntary Churn Rate: Month-to-Month vs. Other Contracts")
ax.set_xticklabels(["False", "True"], rotation=0)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.show()

**Observations:**

- Month-to-month customers churn at 45.75%, versus 6.23% for customers on longer contracts.
- This is consistent with the `contract` breakdown in Section 3, where Month-to-Month (45.75%) is far above One Year (10.71%) and Two Year (2.55%).

**Conclusions:**

- **Verdict: strongly supported.** This is one of the clearest relationships found in this notebook.

**Hypothesis (internet_without_security):** customers with internet service but no online security add-on are more price-sensitive / less
invested, and so churn at a higher rate than internet customers overall.

In [ ]:
internet_customers = df[df["internet_service"] == True]

overall_internet_churn = internet_customers["is_voluntary_churn"].mean() * 100

churn_by_no_security = (
    internet_customers.groupby("internet_without_security")["is_voluntary_churn"].mean() * 100
)

print(f"Overall churn rate among internet customers: {overall_internet_churn:.2f}%")

print()
print("Churn rate (%) by internet_without_security (internet customers only):")
print(churn_by_no_security.round(2))

fig, ax = plt.subplots(figsize=(5, 6))

churn_by_no_security.plot(kind="bar", ax=ax, color="#8172B2")

ax.axhline(
    overall_internet_churn,
    color="black",
    linestyle="--",
    linewidth=1,
    label="overall internet churn rate",
)

ax.set_xlabel("internet_without_security")
ax.set_ylabel("voluntary churn rate (%)")
ax.set_title("Voluntary Churn Rate: Internet Without Security vs. Overall")
ax.set_xticklabels(["False", "True"], rotation=0)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

ax.legend()

plt.tight_layout()
plt.show()

**Observations:**

- Among internet customers, those without the online security add-on churn at 41.68%, compared to 14.57% for those who have it.
- Both figures sit well above, and well below, the overall internet-customer churn rate of 31.75%, respectively.

**Conclusions:**

- **Verdict: supported.** The gap is large enough to look like a genuine relationship rather than noise, though this notebook does not test whether it holds after controlling for other correlated factors (e.g. contract type, tenure).

## 5. Initial Business Insight

In [ ]:
voluntary_churners = df[df["is_voluntary_churn"]]
mtm_churners = voluntary_churners[voluntary_churners["is_month_to_month"]]

reach_fraction = len(mtm_churners) / len(voluntary_churners) * 100
print(f"Total voluntary churners: {len(voluntary_churners)}")
print(f"Of those, on month-to-month contracts: {len(mtm_churners)}")
print(f"Fraction of all voluntary churners reachable via month-to-month-only outreach: {reach_fraction:.2f}%")

**Observations:**

- Month-to-month contracts account for 88.51% of all actual voluntary churners in this dataset (1,649 of 1,863).

**Conclusions:**

- Even this single, simple targeting rule would already reach the large majority of churners, though it would still miss the remaining ~11.5% who churn from longer-term contracts, and it makes no attempt to prioritize within the month-to-month group.
- **This is an informal, exploratory calculation only.** The formal version of this kind of targeting analysis - Precision@K, Lift@K, and the business-facing prioritized outreach list - is locked for later commits (9-11) and is not decided or implemented here.